# Imports

In [55]:
%pip install -q requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement requirements.txt (from versions: none)
ERROR: No matching distribution found for requirements.txt


In [56]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.stattools import adfuller

# Constants

## Control

In [57]:
FETCH_DATA = False
SAVE_RESULT = True

## Dataset

In [58]:
START_DATE = "2024-12-01"
END_DATE = "2025-12-01"
TICKERS_DIR = "./tickers"

TOP_50 = ["BTC", "ETH", "BNB", "XRP", "SOL", "TRX", "DOGE", "ADA", "BCH", "LINK", "RAIN", "XMR", "XLM", "ZEC", "LEO", "LTC", "DAI", "SUI", "AVAX", "HBAR", "SHIB", "NIGHT", "TON", "CRO", "UNI", "DOT", "AAVE", "CC", "BGB", "ASTER", "PI", "ENA", "SKY", "KCS", "WLD", "ONDO", "KAS", "APT", "ARB", "ALGO", "FLR", "ATOM", "FIL", "QNT", "VET", "SET", "M", "CBBTC", "WBT", "PYUSD"]

# Configuration

# Functions

In [59]:
def fetch_from_yfinance(ticker: str, route: str, start_date, end_date, fetch: bool=True):
    if fetch:
        s = ticker.upper().strip()
        yahoo_format = s if s.endswith("-USD") else f"{s}-USD"
        df = yf.download(
            tickers=yahoo_format,
            start=start_date,
            end=end_date,
            auto_adjust=False
        )
        df2 = yf.download(
            tickers=ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False
        )
        if len(df) > 0:
            df.to_csv(route)
            df = pd.read_csv(route, skiprows=[1, 2], header=0)
            df = df.rename(columns={'Price': 'Date'})
            df = df.set_index('Date')
            df.to_csv(route)
        if len(df2) > 0:
            df2.to_csv(route)
            df2 = pd.read_csv(route, skiprows=[1, 2], header=0)
            df2 = df2.rename(columns={'Price': 'Date'})
            df2 = df2.set_index('Date')
            df2.to_csv(route)

    else:
        df = pd.read_csv(route)
    return df

# Fetch data

## Fetch merket data of top 50 crypto

In [60]:
dfs = {}
if not os.path.exists(TICKERS_DIR):
    os.makedirs(TICKERS_DIR)
    print(f"Created directory: {TICKERS_DIR}")
for ticker in TOP_50:
    filename = os.path.join(TICKERS_DIR, f"{ticker}.csv")
    try:
        dfs[ticker] = fetch_from_yfinance(ticker, filename, START_DATE, END_DATE, FETCH_DATA)
    except Exception as e:
        print(f"Failed to fetch {ticker}: {e}")

# Check Stationary

## ADF test

In [61]:
def adf_test(ticker, df, column='Close'):
    if df is None or df.empty or column not in df.columns:
        return {"Ticker": ticker, "Error": "Missing Data"}

    series = df[column].dropna()
    
    if len(series) < 20:
        return {"Ticker": ticker, "Error": f"Insufficient data points: {len(series)}"}

    result = adfuller(series, autolag='AIC')
    
    adf_output = {
        "Ticker": ticker,
        "ADF Statistic": round(result[0], 4),
        "p-value": round(result[1], 4),
        "Stationary": result[1] < 0.05,
        "Lags Used": result[2],
        "Observations": result[3]
    }
    
    return adf_output

In [ ]:
all_results = []

for ticker in TOP_50:
    filename = os.path.join(TICKERS_DIR, f"{ticker}.csv")
    try:
        df = fetch_from_yfinance(ticker, filename, START_DATE, END_DATE, FETCH_DATA)
        
        res = adf_test(ticker, df)
        
        all_results.append(res)
        
        status = "Stationary" if res.get("Stationary") else "Non-Stationary"
        print(f"{ticker}: {status} (p={res.get('p-value')})")
        
    except Exception as e:
        print(f"Failed {ticker}: {e}")

BTC: Non-Stationary (p=0.4028)
ETH: Non-Stationary (p=0.6673)
BNB: Non-Stationary (p=0.7405)
XRP: Non-Stationary (p=None)
SOL: Non-Stationary (p=0.2412)
TRX: Non-Stationary (p=0.9417)
DOGE: Non-Stationary (p=0.063)
ADA: Non-Stationary (p=0.0887)
BCH: Non-Stationary (p=0.9621)
LINK: Non-Stationary (p=0.3089)
RAIN: Non-Stationary (p=0.1023)
XMR: Non-Stationary (p=0.7186)
XLM: Stationary (p=0.0231)
ZEC: Stationary (p=0.0036)
LEO: Non-Stationary (p=0.1793)
LTC: Stationary (p=0.0007)
DAI: Stationary (p=0.0)
SUI: Stationary (p=0.0002)
AVAX: Non-Stationary (p=0.0652)
HBAR: Non-Stationary (p=0.2628)
SHIB: Stationary (p=0.0296)
NIGHT: Stationary (p=0.0058)
TON: Non-Stationary (p=0.2688)
CRO: Non-Stationary (p=0.3248)
UNI: Non-Stationary (p=0.507)
DOT: Stationary (p=0.0001)
AAVE: Non-Stationary (p=0.3775)
CC: Non-Stationary (p=0.1066)
BGB: Non-Stationary (p=0.2843)
ASTER: Non-Stationary (p=0.6278)
PI: Non-Stationary (p=0.5722)
ENA: Non-Stationary (p=0.4508)
SKY: Non-Stationary (p=0.1872)
KCS: No

In [66]:
summary_df = pd.DataFrame(all_results)
summary_df

,Ticker,ADF Statistic,p-value,Stationary,Lags Used,Observations,Error
0,BTC,-1.7554,0.4028,False,0.0,248.0,NaN
1,ETH,-1.2146,0.6673,False,1.0,247.0,NaN
2,BNB,-1.0345,0.7405,False,8.0,356.0,NaN
3,XRP,NaN,NaN,NaN,NaN,NaN,Insufficient data points: 6
4,SOL,-2.1083,0.2412,False,1.0,247.0,NaN
5,TRX,-0.1726,0.9417,False,12.0,236.0,NaN
6,DOGE,-2.7679,0.0630,False,0.0,364.0,NaN
7,ADA,-2.6214,0.0887,False,1.0,363.0,NaN
8,BCH,0.0451,0.9621,False,0.0,248.0,NaN
9,LINK,-1.9500,0.3089,False,5.0,243.0,NaN


## Filter tickers with p-value $\le$ 0.05

In [71]:
stationary_95_df = summary_df[summary_df['p-value'] <= 0.05].copy()
stationary_95_df = stationary_95_df.sort_values(by='p-value')
stationary_95_df

,Ticker,ADF Statistic,p-value,Stationary,Lags Used,Observations,Error
16,DAI,-5.8488,0.0000,True,3.0,361.0,NaN
49,PYUSD,-7.3310,0.0000,True,2.0,362.0,NaN
25,DOT,-4.6399,0.0001,True,6.0,358.0,NaN
17,SUI,-4.5019,0.0002,True,0.0,248.0,NaN
15,LTC,-4.1974,0.0007,True,0.0,248.0,NaN
13,ZEC,-3.7421,0.0036,True,13.0,351.0,NaN
42,FIL,-3.7385,0.0036,True,4.0,360.0,NaN
21,NIGHT,-3.5955,0.0058,True,16.0,299.0,NaN
34,WLD,-3.3193,0.0140,True,2.0,362.0,NaN
12,XLM,-3.1492,0.0231,True,0.0,364.0,NaN
